# LLM setup



In [1]:
!pip uninstall -y datasets


Found existing installation: datasets 2.14.4
Uninstalling datasets-2.14.4:
  Successfully uninstalled datasets-2.14.4


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and installed_version("pandasai") != "3.0.0":
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    # Same versions as uv.lock (torch is preinstalled on Colab).
    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "peft==0.21.2",
        "sentence-transformers==6.1.0",
        "requests",
        "pandasai==3.0.0",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

In [2]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.2 MB/s eta 0:00:00


## Get the project code from github

On Colab, clone your team's repository so that the `ift6758` package can be imported.
For a private repository, create a GitHub [personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) with read access and add it to Colab's *Secrets* (key icon in the left sidebar) as `GITHUB_TOKEN`. Never paste the token in the notebook.

In [3]:
import os
import subprocess
IN_COLAB=True
if IN_COLAB:
    from google.colab import userdata

    REPO = "aliciavaq/data_science_project"
    REPO_DIR = "/content/repo"
    if not os.path.exists(REPO_DIR):
        token = userdata.get("GITHUB_TOKEN")
        subprocess.check_call(["git", "clone", f"https://{token}@github.com/{REPO}.git", REPO_DIR])
    sys.path.insert(0, REPO_DIR)

In [ ]:
!git -C /content/repo pull

Already up to date.


## Check the GPU

In [4]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

GPU: Tesla T4


## Load the model

The first run downloads ~8 GB of weights to the Hugging Face cache.

In [5]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded on cuda:0 (torch.float16)


In [6]:
#test if model interface works
messages = [{"role": "user", "content": "In one sentence, what is a hat trick in hockey?"}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In hockey, a hat trick occurs when a player scores a goal, assists on two other goals, or achieves a combination of these three in a single game.


The function below builds a small example of a game file for the prompt: one play of each shot type and only the players involved in them

In [7]:
import json, requests

game = requests.get("https://api-web.nhle.com/v1/gamecenter/2023021190/play-by-play", timeout=30).json()

def game_example_qwen(game):
    example = {
    "id": game["id"],
    "homeTeam": game["homeTeam"],
    "awayTeam": game["awayTeam"],
    "rosterSpots": [],
    "plays": [],}
    event_types = [ "missed-shot", "blocked-shot", "shot-on-goal", "goal"]
    for play in game["plays"]:
      if play["typeDescKey"] in event_types:
          example["plays"].append(play)
          event_types.remove(play["typeDescKey"])
    ids = set()
    for play in example["plays"]:
        for key, value in play.get("details", {}).items():
            if key.endswith("PlayerId") or key == "goalieInNetId":
                ids.add(value)
    example["rosterSpots"] = [p for p in game["rosterSpots"] if p["playerId"] in ids]
    return example

example_json = json.dumps(game_example_qwen(game), indent=1)
print(len(tokenizer(example_json)["input_ids"]), "tokens")
print(example_json)

2156 tokens
{
 "id": 2023021190,
 "homeTeam": {
  "id": 8,
  "commonName": {
   "default": "Canadiens"
  },
  "abbrev": "MTL",
  "score": 5,
  "sog": 38,
  "logo": "https://assets.nhle.com/logos/nhl/svg/MTL_light.svg",
  "darkLogo": "https://assets.nhle.com/logos/nhl/svg/MTL_dark.svg",
  "placeName": {
   "default": "Montr\u00e9al"
  },
  "placeNameWithPreposition": {
   "default": "Montr\u00e9al",
   "fr": "de Montr\u00e9al"
  }
 },
 "awayTeam": {
  "id": 13,
  "commonName": {
   "default": "Panthers"
  },
  "abbrev": "FLA",
  "score": 3,
  "sog": 40,
  "logo": "https://assets.nhle.com/logos/nhl/svg/FLA_light.svg",
  "darkLogo": "https://assets.nhle.com/logos/nhl/svg/FLA_dark.svg",
  "placeName": {
   "default": "Florida",
   "fr": "Floride"
  },
  "placeNameWithPreposition": {
   "default": "Florida",
   "fr": "de la Floride"
  }
 },
 "rosterSpots": [
  {
   "teamId": 13,
   "playerId": 8475765,
   "firstName": {
    "default": "Vladimir"
   },
   "lastName": {
    "default": "Tarase

# Generating the cleaning code using qwen

In [8]:
code_prompt = """
TASK: Write a Python function `create_shot_dataframe(year: int, path: str) -> pandas.DataFrame`
that turns raw NHL play-by-play JSON files for one season into a  pandas DataFrame.
Each game file has the structure shown in the EXAMPLE JSON FILE below (simplified).

SPECIFICATIONS:
- Files are stored at `{path}/{season}/{game_id}.json`.Here 'season' is given by the code  f"{year}-{str(year + 1)[-2:]}"
  Raise FileNotFoundError if the season folder does not exist.
- Read the files with json and use pandas for the final DataFrame.
- Keep only plays whose typeDescKey is "shot-on-goal" or "goal".IGNORE all other plays
- One row per play, uniquely identified by (game_id, event_id). Raise ValueError if there are duplicates.
- the Columns of the dataframe should be : game_id (int), event_id, season, game_type, period, time (MM:SS), team (abbreviation,example :"MTL"),
  event_type, x, y, shooter (full name), goalie (full name), shot_type, is_empty_net, situation_code.
- Keep missing values as None. Never replace them with 0 or False.

IMPORTANT:
- Most event information is found inside the play's "details".
- For goals, the shooter is the scorer.
- game_type comes from the game ID: digits 5 and 6 are "02" for regular season and "03" for playoffs.
- situationCode has 4 digits: away goalie (1/0), away skaters, home skaters, home goalie (1/0).
  A goalie digit of 0 means that net is empty.

Reply with only one ```python code block, with a short docstring and minimal comments.
"""
code_prompt = code_prompt + "EXAMPLE JSON FILE=\n```json\n" + example_json + "\n```\n"
print(code_prompt)


TASK: Write a Python function `create_shot_dataframe(year: int, path: str) -> pandas.DataFrame`
that turns raw NHL play-by-play JSON files for one season into a  pandas DataFrame.
Each game file has the structure shown in the EXAMPLE JSON FILE below (simplified).

SPECIFICATIONS:
- Files are stored at `{path}/{season}/{game_id}.json`.Here 'season' is given by the code  f"{year}-{str(year + 1)[-2:]}" 
  Raise FileNotFoundError if the season folder does not exist.
- Read the files with json and use pandas for the final DataFrame.
- Keep only plays whose typeDescKey is "shot-on-goal" or "goal".IGNORE all other plays
- One row per play, uniquely identified by (game_id, event_id). Raise ValueError if there are duplicates.
- the Columns of the dataframe should be : game_id (int), event_id, season, game_type, period, time (MM:SS), team (abbreviation,example :"MTL"),
  event_type, x, y, shooter (full name), goalie (full name), shot_type, is_empty_net, situation_code.
- Keep missing values as 

In [9]:
from pathlib import Path
from ift6758.llm.qwen import ask_qwen

generated_code = ask_qwen(model, tokenizer, code_prompt,
               system="You are an expert Python data engineer.",
               max_new_tokens=1500)



[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [10]:
from IPython.display import  Markdown
display(Markdown(generated_code))


```python
import json
import pandas as pd
import os
from pathlib import Path

def create_shot_dataframe(year: int, path: str) -> pd.DataFrame:
    season_folder = f"{year}-{str(year + 1)[-2:]}"
    season_path = Path(path) / season_folder
    
    if not season_path.exists():
        raise FileNotFoundError(f"Season folder {season_folder} not found at {path}")
    
    all_plays = []
    
    for game_file in season_path.glob("*.json"):
        with open(game_file, 'r') as f:
            data = json.load(f)
        
        game_id = data['id']
        game_type = data['id'] // 100000 % 100  # digits 5-6: 02 for regular, 03 for playoffs
        home_team_abbrev = data['homeTeam']['abbrev']
        away_team_abbrev = data['awayTeam']['abbrev']
        
        for play in data['plays']:
            if play['typeDescKey'] not in ['shot-on-goal', 'goal']:
                continue
                
            event_id = play['eventId']
            period = play['periodDescriptor']['number']
            time_str = play['timeInPeriod']
            situation_code = play['situationCode']
            
            # Extract details
            details = play['details']
            x, y = details.get('xCoord'), details.get('yCoord')
            shooter_id = details.get('shootingPlayerId')
            goalie_id = details.get('goalieInNetId')
            event_owner_team_id = details.get('eventOwnerTeamId')
            
            # Determine team and shooter/goalie names
            team_abbrev = home_team_abbrev if event_owner_team_id == data['homeTeam']['id'] else away_team_abbrev
            
            # Shooter name
            shooter_name = None
            goalie_name = None
            
            if play['typeDescKey'] == 'goal':
                # For goals, shooter is scorer
                scoring_player_id = details.get('scoringPlayerId')
                shooter_name = next((p for p in data['rosterSpots'] 
                                    if p['playerId'] == scoring_player_id), None)
            else:
                # For shot-on-goal, shooter is from shootingPlayerId
                shooter_name = next((p for p in data['rosterSpots'] 
                                    if p['playerId'] == shooter_id), None)
            
            # Goalie name
            goalie_name = next((p for p in data['rosterSpots'] 
                                if p['playerId'] == goalie_id and p['positionCode'] == 'G'), None)
            
            # Shot type and empty net
            shot_type = details.get('shotType', None)
            is_empty_net = (situation_code[0] == '0')  # goalie digit 0 means empty net
            
            # Build row
            row = {
                'game_id': game_id,
                'event_id': event_id,
                'season': season_folder,
                'game_type': f"{game_type:02d}",
                'period': period,
                'time': time_str,
                'team': team_abbrev,
                'event_type': play['typeDescKey'],
                'x': x,
                'y': y,
                'shooter': shooter_name['firstName']['default'] + ' ' + shooter_name['lastName']['default'] if shooter_name else None,
                'goalie': goalie_name['firstName']['default'] + ' ' + goalie_name['lastName']['default'] if goalie_name else None,
                'shot_type': shot_type,
                'is_empty_net': is_empty_net,
                'situation_code': situation_code
            }
            all_plays.append(row)
    
    df = pd.DataFrame(all_plays)
    
    # Ensure unique (game_id, event_id)
    if df.duplicated(subset=['game_id', 'event_id']).any():
        raise ValueError("Duplicate (game_id, event_id) entries found")
    
    return df
```

###Generated code with manual corrections

Note: Unfortunately, we were unable to add the strength_type attribute as a column. It requires too many details to be added to prompt which causes more hallucination and more error in the generated code. After 5 unsuccesful attempts,we choose to not specify strength type in prompt and just require situation code. Instead, as a manual correction, we reuse the helper `get_strength` from manual code.

In [25]:
import json
import pandas as pd
import os
from pathlib import Path
from ift6758.data.question_3 import get_strength  # manual fix 7: reuse manual strength helper

def create_shot_dataframe(year: int, path: str) -> pd.DataFrame:
    season_folder = f"{year}-{str(year + 1)[-2:]}"
    season_path = Path(path) / season_folder

    if not season_path.exists():
        raise FileNotFoundError(f"Season folder {season_folder} not found at {path}")

    all_plays = []

    for game_file in season_path.glob("*.json"):
      # manual fix 1: encoding
       # with open(game_file, 'r') as f:
        with open(game_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        game_id = data['id']
        #game_type = data['id'] // 100000 % 100  # digits 5-6: 02 for regular, 03 for playoffs

        # manual fix 2: divide by 10000 (not 100000) and return labels
        game_type = {2: "Regular season", 3: "Playoffs"}.get(data['id'] // 10000 % 100)

        home_team_abbrev = data['homeTeam']['abbrev']
        away_team_abbrev = data['awayTeam']['abbrev']

        for play in data['plays']:
            if play['typeDescKey'] not in ['shot-on-goal', 'goal']:
                continue

            event_id = play['eventId']
            period = play['periodDescriptor']['number']
            time_str = play['timeInPeriod']
            situation_code = play['situationCode']

            # Extract details
            details = play['details']
            x, y = details.get('xCoord'), details.get('yCoord')
            shooter_id = details.get('shootingPlayerId')
            goalie_id = details.get('goalieInNetId')
            event_owner_team_id = details.get('eventOwnerTeamId')
            # manual fix 3: None when the team matches neither home nor away
            if event_owner_team_id == data['homeTeam']['id']:
                team_abbrev = home_team_abbrev
            elif event_owner_team_id == data['awayTeam']['id']:
                team_abbrev = away_team_abbrev
            else:
                team_abbrev = None

            # Determine team and shooter/goalie names
            #team_abbrev = home_team_abbrev if event_owner_team_id == data['homeTeam']['id'] else away_team_abbrev

            # Shooter name
            shooter_name = None
            goalie_name = None

            if play['typeDescKey'] == 'goal':
                # For goals, shooter is scorer
                scoring_player_id = details.get('scoringPlayerId')
                shooter_name = next((p for p in data['rosterSpots']
                                    if p['playerId'] == scoring_player_id), None)
            else:
                # For shot-on-goal, shooter is from shootingPlayerId
                shooter_name = next((p for p in data['rosterSpots']
                                    if p['playerId'] == shooter_id), None)

            # Goalie name
            goalie_name = next((p for p in data['rosterSpots']
                                if p['playerId'] == goalie_id and p['positionCode'] == 'G'), None)

            # Shot type and empty net
            shot_type = details.get('shotType', None)
            #is_empty_net = (situation_code[0] == '0')  # goalie digit 0 means empty net
            # manual fix 4: check the goalie of the team that did NOT shoot
            if team_abbrev == home_team_abbrev:
                is_empty_net = situation_code[0] == '0'
            elif team_abbrev == away_team_abbrev:
                is_empty_net = situation_code[3] == '0'
            else:
                is_empty_net = None
            # manual fix 7
            strength_type, _ = get_strength(
                situation_code, event_owner_team_id,
                data['homeTeam']['id'], data['awayTeam']['id']
            )

            # Build row
            row = {
                'game_id': game_id,
                'event_id': event_id,
                'season': season_folder,
                'game_type': game_type,  # manual fix 2: label instead of f"{game_type:02d}"
                'period': period,
                'time': time_str,
                'team': team_abbrev,
                'home_team': home_team_abbrev,  # manual fix 5: this will be needed by add_strength in later function
                'event_type': play['typeDescKey'],
                'x': x,
                'y': y,
                'shooter': shooter_name['firstName']['default'] + ' ' + shooter_name['lastName']['default'] if shooter_name else None,
                'goalie': goalie_name['firstName']['default'] + ' ' + goalie_name['lastName']['default'] if goalie_name else None,
                'shot_type': shot_type,
                'is_empty_net': is_empty_net,
                'situation_code': situation_code,
                'strength_type': strength_type  # manual fix 7
            }

            all_plays.append(row)

    df = pd.DataFrame(all_plays)

    # Ensure unique (game_id, event_id)
    #if df.duplicated(subset=['game_id', 'event_id']).any():
    #   raise ValueError("Duplicate (game_id, event_id) entries found")
    # manual fix 6: don't crash when there are no events
    if not df.empty and df.duplicated(subset=['game_id', 'event_id']).any():
        raise ValueError("Duplicate (game_id, event_id) entries found")


    return df